# RAG Evaluation Test Set Generation

This example shows how to use the [Ragas](https://docs.ragas.io/en/stable/) (```v 0.1.22```) framework to generate a **test set** that can be used to evaluate the quality of a RAG pipeline. We then use the Python [LangChain](https://python.langchain.com/docs/introduction/) library to run some requests through this pipeline and we evaluate the quality of the results.

### <u>Requirements</u>
1. As you will accessing the LLMs and embedding models through Vector AI Engineering's Kaleidoscope Service (Vector Inference + Autoscaling), you will need to request a KScope API Key:

      Run the following command (replace ```<user_id>``` and ```<password>```) from **within the cluster** to obtain the API Key. The ```access_token``` in the output is your KScope API Key.
  ```bash
  curl -X POST -d "grant_type=password" -d "username=<user_id>" -d "password=<password>" https://kscope.vectorinstitute.ai/token
  ```
2. After obtaining the `.env` configurations, make sure to create the ```.kscope.env``` file in your home directory (```/h/<user_id>```) and set the following env variables:
- For local models through Kaleidoscope (KScope):
    ```bash
    export OPENAI_BASE_URL="https://kscope.vectorinstitute.ai/v1"
    export OPENAI_API_KEY=<kscope_api_key>
    ```
- For OpenAI models:
   ```bash
   export OPENAI_BASE_URL="https://api.openai.com/v1"
   export OPENAI_API_KEY=<openai_api_key>
   ```

## Set up the RAG workflow environment

#### Import libraries

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
#!pip install pdfplumber


In [4]:
#import pdfplumber

In [5]:
import numpy as np
import os
import sys

from datasets import Dataset
from pathlib import Path

from langchain.chains import RetrievalQA
from langchain.document_loaders.pdf import PyPDFDirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from ragas import evaluate
from ragas.metrics import Faithfulness, ContextPrecision, AnswerCorrectness
from ragas.testset import TestsetGenerator
from ragas.testset.evolutions import simple, reasoning, multi_context

#### Load config files

In [6]:
# Add root folder of the rag_bootcamp repo to PYTHONPATH
current_dir = Path().resolve()
parent_dir = current_dir.parent
sys.path.insert(0, str(parent_dir))



In [7]:
from utils.load_secrets import load_env_file
load_env_file()

#### Set up some helper functions

In [8]:
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [f"Document {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

#### Make sure other necessary items are in place

In [9]:
# Look for the source_documents folder and make sure there is at least 1 pdf file here
contains_pdf = False
documents_path = "./source_documents"
if not os.path.exists(documents_path):
    print(f"ERROR: The {documents_path} subfolder must exist under this notebook")
for filename in os.listdir(documents_path):
    contains_pdf = True if ".pdf" in filename else contains_pdf
if not contains_pdf:
    print(f"ERROR: The {documents_path} subfolder must contain at least one .pdf file")

## Generate a sythentic test set

#### Start by loading in the documents we'll be using to augment our RAG generations

In [10]:
# skip the next step

In [11]:
#loader = PyPDFDirectoryLoader(documents_path)
#documents = loader.load()
#for document in documents:
#    document.metadata['file_name'] = document.metadata['source']
    
#%%time
# Load the IBIS pdfs
#directory_path = "./source_documents"
directory_path = "/projects/RAG2/scotia-2/Datasets-Scotia-2/IBIS"
loader = PyPDFDirectoryLoader(directory_path)
docs = loader.load()
print(f"Number of source documents: {len(docs)}")
for document in docs:
    document.metadata['file_name'] = document.metadata['source']
# Split the documents into smaller chunks
#text_splitter = RecursiveCharacterTextSplitter(chunk_size=3600, chunk_overlap=32)
#chunks = text_splitter.split_documents(docs)
#print(f"Number of text chunks: {len(chunks)}")

Number of source documents: 280


In [12]:

from langchain.document_loaders import TextLoader

In [13]:
directory_path  ='/projects/RAG2/scotia-2/Datasets-Scotia-2/PDF_text'

documents=[]
for filename in os.listdir(directory_path):
    if filename.endswith('.txt'):
        file_path = os.path.join(directory_path, filename)
        print (file_path)
        loader = TextLoader(file_path)

        # Load the document

        document = loader.load()
        documents = documents+ document
        #chunks2 =text_splitter.split_documents(document)
        #print(f"Number of text chunks: {len(chunks2)}")
        #chunks= chunks +chunks2

/projects/RAG2/scotia-2/Datasets-Scotia-2/PDF_text/11114CA Wheat Farming in Canada Industry Report.txt
/projects/RAG2/scotia-2/Datasets-Scotia-2/PDF_text/48422CA Local Specialized Freight Trucking in Canada Industry Report.txt
/projects/RAG2/scotia-2/Datasets-Scotia-2/PDF_text/44111CA New Car Dealers in Canada Industry Report.txt
/projects/RAG2/scotia-2/Datasets-Scotia-2/PDF_text/48412CA Long-Distance Freight Trucking in Canada Industry Report.txt
/projects/RAG2/scotia-2/Datasets-Scotia-2/PDF_text/33639CA Auto Parts Manufacturing in Canada Industry Report.txt
/projects/RAG2/scotia-2/Datasets-Scotia-2/PDF_text/48423CA Long-Distance Specialized Freight Trucking in Canada Industry .txt
/projects/RAG2/scotia-2/Datasets-Scotia-2/PDF_text/11115CA Corn Farming in Canada Industry Repor.txt


In [14]:
type(document)

list

#### Now use OpenAI to generate a test set from the data in these documents (This takes about 2-3 minutes)

**IMP Note:** The LLM and embedding model used for test set generation should be more capable than the model being evaluated. Hence, we will use OpenAI GPT-4o and OpenAI embeddings for this purpose.

Store your OpenAI API key in ```~/.ragas_openai.env``` using the following format (this is in addition to ```~/.kscope.env```):

```bash
export RAGAS_OPENAI_BASE_URL="https://api.openai.com/v1"
export RAGAS_OPENAI_API_KEY=<openai_api_key>
```

In [15]:
from utils.load_secrets import load_env_file_ragas
load_env_file_ragas()

In [16]:
#os.environ["RAGAS_OPENAI_API_KEY"]

In [17]:
os.environ["RAGAS_OPENAI_BASE_URL"]

'https://api.openai.com/v1'

In [37]:
generator_llm = ChatOpenAI(
    model="gpt-4o-mini",
    base_url=os.environ["RAGAS_OPENAI_BASE_URL"],
    api_key=os.environ["RAGAS_OPENAI_API_KEY"],
)
generator_embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    base_url=os.environ["RAGAS_OPENAI_BASE_URL"],
    api_key=os.environ["RAGAS_OPENAI_API_KEY"],
)

In [20]:
# skip the next line

In [35]:
%%time
#generator_llm = (
#    model="DeepSeek-R1-Distill-Llama-8B",
#    base_url=os.environ["OPENAI_BASE_URL"],
#    api_key=os.environ["OPENAI_API_KEY"],
#)
# Define the RAG embeddings model (different than the OpenAI embedding model defined above for test set generation)
model_kwargs = {'device': 'cuda', 'trust_remote_code': True}
encode_kwargs = {'normalize_embeddings': True} # set True to compute cosine similarity
print(f"Setting up the RAG LLM...")
llm = ChatOpenAI(
    #model="DeepSeek-R1-Distill-Llama-8B",
    model="Meta-Llama-3.1-8B-Instruct",
    temperature=0,
    max_tokens=256,
    base_url=os.environ["OPENAI_BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)

Setting up the RAG LLM...
CPU times: user 369 ms, sys: 228 ms, total: 597 ms
Wall time: 1.18 s


In [19]:
#generator_llm.generate('what is  the day today?, Answer in no more then 10 words')

In [109]:
%%time
# Create generator with OpenAI model
generator = TestsetGenerator.from_langchain(
    generator_llm=generator_llm,
    critic_llm=generator_llm,
    embeddings=generator_embeddings,
)

# Generate the test set
testset = generator.generate_with_langchain_docs(
    documents=documents, 
    test_size=50,
    distributions={simple: 0.5, reasoning: 0.25, multi_context: 0.25},
)

embedding nodes:   0%|          | 0/260 [00:00<?, ?it/s]

Filename and doc_id are the same for all nodes.


Generating:   0%|          | 0/50 [00:00<?, ?it/s]

IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



CPU times: user 24.4 s, sys: 643 ms, total: 25 s
Wall time: 3min 33s


In [18]:
%%time
# Create generator with OpenAI model
generator = TestsetGenerator.from_langchain(
    generator_llm=generator_llm,
    critic_llm=generator_llm,
    embeddings=generator_embeddings,
)

# Generate the test set
testset = generator.generate_with_langchain_docs(
    documents=documents, 
    test_size=1,
    distributions={simple: 0.5, reasoning: 0.25, multi_context: 0.25},
)

embedding nodes:   0%|          | 0/260 [00:00<?, ?it/s]

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

#### Preview the test dataset so far

In [19]:
testset1 = testset.to_pandas()

In [20]:
# testset1.to_parquet ('../../Testing_Data/testset_llama.parquet')
# testset1.to_csv ('../../Testing_Data/testset_llama.csv', index =False)

In [20]:
import joblib

In [110]:


# Save (serialize) the object to a file
joblib.dump(testset, "../../Testing_Data/testset_openai_clean_mini.pkl")


['../../Testing_Data/testset_openai_clean_mini.pkl']

In [22]:
testset1.to_parquet ('../../Testing_Data/testset_openai_clean.parquet')
testset1.to_csv ('../../Testing_Data/testset_openai_clean.csv', index =False)

In [21]:
#load 
# Load (deserialize) the object from file
testset = joblib.load("../../Testing_Data/testset_openai_clean.pkl")


## Now, start the RAG pipeline!

#### Choose the RAG LLM and embedding model
Note: This is different than the OpenAI LLM and embedding model defined above for test set generation.

In [25]:
RAG_LLM_MODEL_NAME = "Meta-Llama-3.1-8B-Instruct"
RAG_EMBEDDING_MODEL_NAME = "BAAI/bge-base-en-v1.5"

#### Generate answers for all the questions in our test set

Go through the embedding, storage and retrieval steps.

In [23]:
# Split the documents into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=3000, chunk_overlap=32)  ### Change Chunks here 
chunks = text_splitter.split_documents(documents)
print(f"Number of text chunks: {len(chunks)}")

Number of text chunks: 193


In [26]:
# Define the RAG embeddings model (different than the OpenAI embedding model defined above for test set generation)
model_kwargs = {'device': 'cuda', 'trust_remote_code': True}
encode_kwargs = {'normalize_embeddings': True} # set True to compute cosine similarity

print(f"Setting up the RAG embeddings model...")
embeddings = HuggingFaceEmbeddings(
    model_name=RAG_EMBEDDING_MODEL_NAME,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)
print (f"setting up {RAG_EMBEDDING_MODEL_NAME}")

Setting up the RAG embeddings model...
setting up BAAI/bge-base-en-v1.5


In [27]:
# Create the vector store and the retriever
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [28]:
RAG_LLM_MODEL_NAME

'Meta-Llama-3.1-8B-Instruct'

In [29]:
%%time
# Define the RAG LLM (different than the OpenAI LLM defined above for test set generation)
print(f"Setting up the RAG LLM...")
llm = ChatOpenAI(
    model=RAG_LLM_MODEL_NAME,
    temperature=0,
    max_tokens=256,
    base_url=os.environ["OPENAI_BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)
print (f"setting up {RAG_LLM_MODEL_NAME}")

Setting up the RAG LLM...
setting up Meta-Llama-3.1-8B-Instruct
CPU times: user 24 ms, sys: 3.58 ms, total: 27.6 ms
Wall time: 24.2 ms


Iterate over the questions in our synthetic testset, and run them each through the RAG pipeline to see what answers get returned. (This also takes 2-3 minutes)

In [102]:
# %%time
# #USe OpenAi

# dataset = testset.to_dataset()
# answers = np.empty(len(dataset), dtype=object)

# for index, row in enumerate(dataset):
#     query = row["question"]
    
#     # Run the query through the RAG pipeline
#     rag_pipeline = RetrievalQA.from_llm(
#         llm=generator_llm,
#         retriever=retriever
#     )
#     answer = rag_pipeline.invoke(input=query)
#     answer = answer["result"]
#     print(f"Result {index}\nQuestion: {query}\nAnswer: {answer}\n")
    
#     # Store the result
#     answers[index] = answer

In [111]:
%%time
#Use LLama
dataset = testset.to_dataset()
answers = np.empty(len(dataset), dtype=object)

for index, row in enumerate(dataset):
    query = row["question"]
    
    # Run the query through the RAG pipeline
    rag_pipeline = RetrievalQA.from_llm(
        llm=llm,
        retriever=retriever
    )
    answer = rag_pipeline.invoke(input=query)
    answer = answer["result"]
    #print(f"Result {index}\nQuestion: {query}\nAnswer: {answer}\n")
    print (index)
    
    # Store the result
    answers[index] = answer

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
CPU times: user 2.15 s, sys: 131 ms, total: 2.28 s
Wall time: 6min 4s


Add the list of answers into our original dataset. Now we have a complete test set that is ready for evaluation.

In [112]:
dataset = dataset.add_column("answer", answers)

In [113]:
type(dataset)

datasets.arrow_dataset.Dataset

## Evaluate the results

#### Preview the final test set

In [114]:
dataset.to_pandas().head(5)

,question,contexts,ground_truth,evolution_type,metadata,episode_done,answer
0,What impact does the consumer confidence index...,"[ kilograms (14,000 pounds).\nECOBOOST ENGINE\...",The consumer confidence index impacts new car ...,simple,[{'source': '/projects/RAG2/scotia-2/Datasets-...,True,"According to the context, consumer confidence ..."
1,What are the key statistics for long-distance ...,"[154 14.5 1.0 33.7\n2026 275,063 0.5 1.7 1.7 4...",The key statistics for long-distance freight t...,simple,[{'source': '/projects/RAG2/scotia-2/Datasets-...,True,"Based on the provided context, the key statist..."
2,What role does after-sales service play in the...,[ attract customers. Dealerships often partner...,Provision of superior after-sales service is a...,simple,[{'source': '/projects/RAG2/scotia-2/Datasets-...,True,After-sales service plays a crucial role in th...
3,What factors contribute to the dominance of co...,[.8m\nExports High Increasing\nWhat are the in...,The dominance of corn farming in Ontario and Q...,simple,[{'source': '/projects/RAG2/scotia-2/Datasets-...,True,"According to the context, the factors that con..."
4,What role does competitive pricing play in the...,[ and maintaining an open line of communicatio...,Competitive pricing plays a crucial role in th...,simple,[{'source': '/projects/RAG2/scotia-2/Datasets-...,True,"According to the provided context, competitive..."


Run the evaluation query to score the results. In this evaluation, we are looking at the following metrics:
- *[Faithfulness](https://docs.ragas.io/en/v0.1.21/concepts/metrics/faithfulness.html)*: Are all the claims that are made in the answer inferred from the given context(s)?
- *[Context Precision](https://docs.ragas.io/en/v0.1.21/concepts/metrics/context_precision.html)*: Did our retriever return good results that matched the question it was being asked?
- *[Answer Correctness](https://docs.ragas.io/en/v0.1.21/concepts/metrics/answer_correctness.html)*: Was the generated answer correct? Was it complete?

In [115]:
%%time
score = evaluate(
    dataset=dataset,
    metrics=[
        Faithfulness(),
        ContextPrecision(),
        AnswerCorrectness(),
    ],
    #llm=llm, # Using LLAma LLM as the evaluator
     llm=generator_llm, # Using OpenAI LLM as the evaluator
    embeddings=embeddings,
)


Evaluating:   0%|          | 0/147 [00:00<?, ?it/s]

No statements were generated from the answer.
No statements were generated from the answer.
No statements were generated from the answer.


CPU times: user 23.4 s, sys: 272 ms, total: 23.7 s
Wall time: 2min 34s


In [108]:
score.to_pandas().head(15)

,question,contexts,ground_truth,evolution_type,metadata,episode_done,answer,faithfulness,context_precision,answer_correctness
0,What impact does economic volatility have on t...,[ Decline\n*Growth is based on change in share...,Economic volatility causes volatility in the d...,simple,[{'source': '/projects/RAG2/scotia-2/Datasets-...,True,"According to the context, economic volatility ...",1.0,1.0,0.983133


In [118]:
score["answer_correctness"].mean()

0.6117151574182814

In [119]:
score["faithfulness"].mean()

0.6423816752052383

In [45]:
#score.to_pandas().to_parquet  ("../../Testing_Data/score_llama_answeropenAi.parquet")

In [68]:
df = score.to_pandas().copy()

In [92]:
df = df.drop(['metadata', 'episode_done'], axis =1 )

In [93]:
df[df.answer_correctness<0.5]

,question,contexts,ground_truth,evolution_type,answer,faithfulness,context_precision,answer_correctness
1,Here is a question that can be fully answered ...,[ navigate economic downturns and invest in ne...,The government's pro-immigration policies are ...,simple,The expected outcomes of the government's pro-...,1.000000,1.0,0.469651
4,Here is a question that can be fully answered ...,[.8 1.9 2.6 1.6 3.2 2.8 2.5\nLabor intensive C...,The answer to given question is not present in...,simple,"To calculate the average debt ratio, we need t...",0.500000,1.0,0.104300
9,Here is a question that can be fully answered ...,[ who don't enjoy the same pricing power.\nâ¢...,The expected long-term effects of rising truck...,simple,"According to the context, the expected long-te...",0.900000,1.0,0.488594
10,Here is a question that can be fully answered ...,[.8m\nExports High Increasing\nWhat are the in...,"Ireland, the US, and the United Kingdom are ke...",simple,"According to the context, the key internationa...",1.000000,1.0,0.235318
11,Here is a question that can be fully answered ...,[ navigate economic downturns and invest in ne...,nan,simple,"Based on the provided context, here are the ex...",0.812500,1.0,0.107648
12,Here is a question that can be fully answered ...,[s Low Revenue Growth (2005-2024)\nLow Outlier...,The answer to given question is present in con...,simple,"According to the context, the expected Compoun...",1.000000,1.0,0.114328
14,Here is a question that can be fully answered ...,"[154 14.5 1.0 33.7\n2026 275,063 0.5 1.7 1.7 4...",The answer to given question is present in con...,simple,"According to the context, the number of enterp...",0.500000,1.0,0.112525
16,Here is a question that can be fully answered ...,"[154 14.5 1.0 33.7\n2026 275,063 0.5 1.7 1.7 4...",The average annual percentage change in the nu...,simple,To calculate the average annual percentage cha...,0.000000,1.0,0.171662
20,Here is a question that can be fully answered ...,[Building strong ties with suppliers ensures a...,Low barriers to entry promote a fiercely compe...,simple,Low barriers to entry in the long-distance fre...,0.375000,1.0,0.351872
22,Here is a question that can be fully answered ...,[ market. More prominent companies will target...,Developing a strong reputation is the primary ...,simple,"According to the context, the primary way in w...",0.500000,1.0,0.423577


In [73]:
from collections import Counter

In [97]:
def print_score(df, idx):
    print (score.to_pandas().iloc[idx].question)
    print ('='*120)
    print (score.to_pandas().iloc[idx].contexts)
    print ('='*120)
    print (score.to_pandas().iloc[idx].ground_truth)
    print ('='*120)
    print (score.to_pandas().iloc[idx].answer)

In [99]:
print_score(df, 27)

Here is a rewritten version of the question:

"When did Canada's Auto Parts Mfg. industry see a big revenue boost?"

I've used abbreviations (e.g. "Mfg." for "Manufacturing") and shortened the question to make it more concise while still conveying the same meaning.
['9 2,855.5 18,081.5 1,041.7\n2016 9,346.4 1,769.8 377 345 12,156 2,879.1 19,199.2 935.0\n2017 9,059.6 2,395.1 383 351 16,844 2,749.9 19,197.7 1,210.0\n2018 8,616.7 2,037.0 374 343 17,181 2,931.9 18,415.3 1,237.3\n2019 7,475.2 1,727.5 371 341 15,017 3,025.1 18,695.4 1,054.7\n2020 6,360.4 1,331.5 368 338 12,685 2,619.6 14,355.8 802.4\n2021 6,731.1 1,641.8 363 333 14,725 2,994.8 13,359.0 903.1\n2022 7,583.2 1,775.0 365 334 15,940 3,557.3 14,579.2 982.6\n2023 7,766.0 1,845.6 379 346 16,429 3,663.2 17,205.4 1,011.4\n2024 7,915.7 1,920.0 382 349 16,833 3,737.5 17,433.1 1,035.2\n2025 8,079.7 1,954.3 385 351 17,088 3,796.1 18,338.2 1,052.0\n2026 8,225.6 1,990.1 389 354 17,323 3,904.5 17,534.1 1,067.4\n2027 8,319.3 2,014.8 392 357 1

In [42]:
score.to_pandas().to_csv  ("../../Testing_Data/score_llama_answer_LLAMA_scoreOpenAi_mini.csv")

In [94]:
#"Meta-Llama-3.1-8B-Instruct"
#chunksize, answer correctness, faithfulness, number of question
#10000, 0.592, 0.698, 50
# 5000, 0.617, 0.700, 50
# 3500, 0.677, 0.734, 50
# 3000, 0.688, 0.800, 50
# 2000, 0.629, 0.742, 50
# 1000, 0.644, 0.703, 50
#  500, 0.619, 0.503, 50
#  250, 0.594, 0.527, 50



In [ ]:
#2500, 0.62, 0.825
#3000, 0.60, 0.78 llama, llamaa, llama, llama
#3000, 062, 0.73, llama, llama, openAi, llama

#0.61, 0.64   openAi, llama,  llama, openAi, evaluatio  is mini, dataset is not right